In [1]:
!py -m pip install pandas tqdm dnspython ipwhois


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from tqdm import tqdm  # opcional: progresso
import dns.resolver
from ipwhois import IPWhois
import os


companies_info_df = pd.read_csv('.\companies_ranked_by_revenue.csv') 

dns_pipeline_df = companies_info_df[['domain']]

import dns.resolver
from ipwhois import IPWhois


def extract_org_name(rdap_result):
    """
    Extração robusta do nome da organização a partir da resposta RDAP.

    """
    for obj in rdap_result.get("objects", {}).values():
        contact = obj.get("contact", {})
        roles = obj.get("roles", [])
        if "registrant" in roles or contact.get("kind") == "group":
            name = contact.get("name")
            if name:
                return name.replace('"', '').replace("'", "")
    return None

import dns.resolver
from ipwhois import IPWhois
import os

import dns.resolver
from dns.resolver import NXDOMAIN, NoAnswer, LifetimeTimeout

def get_ns_info(domain: str):
    """
    Retorna sempre (ns_data, failed_entry_or_none).

    ns_data: lista de dicts com chaves "NS" e "NS-A" (máx. 3 NS).
    failed_entry_or_none: dict com info do erro e domínio original (ou None).
    """
    domain = str(domain).strip().rstrip('.')
    ns_data = []
    failed_entry = None

    try:
        # lifetime pequeno para não travar
        ns_answers = dns.resolver.resolve(domain, 'NS', lifetime=3.0)
        for ns in ns_answers[:3]:
            ns_name = str(ns.target).rstrip('.')
            ns_ip = None
            # tenta IPv4; se falhar, tenta IPv6
            try:
                a_answers = dns.resolver.resolve(ns_name, 'A', lifetime=3.0)
                ns_ip = a_answers[0].to_text()
            except Exception:
                try:
                    aaaa_answers = dns.resolver.resolve(ns_name, 'AAAA', lifetime=3.0)
                    ns_ip = aaaa_answers[0].to_text()
                except Exception:
                    pass

            ns_data.append({
                "NS": ns_name,
                "NS-A": ns_ip,
            })

    except Exception as e:
        # tenta "podar" o domínio uma vez (ex.: sub.example.com -> example.com)
        trimmed_domain = domain.split('.', 1)[1] if '.' in domain else None

        # registra o erro no domínio original
        failed_entry = {
            "domain": domain,
            "error": f"{type(e).__name__}: {e}",
            "trimmed_domain": trimmed_domain
        }

        if trimmed_domain:
            try:
                # chama recursivamente e só reaproveita a lista de NS do domínio podado
                trimmed_ns_data, _ = get_ns_info(trimmed_domain)
                ns_data = trimmed_ns_data
            except Exception as e2:
                # se até o podado falhar, anota erro secundário
                failed_entry["secondary_error"] = f"{type(e2).__name__}: {e2}"

    return ns_data, failed_entry

        
dns_pipeline_df.head()

<>:8: SyntaxWarning: invalid escape sequence '\c'
<>:8: SyntaxWarning: invalid escape sequence '\c'
C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_14820\794760063.py:8: SyntaxWarning: invalid escape sequence '\c'
  companies_info_df = pd.read_csv('.\companies_ranked_by_revenue.csv')


,domain
0,pg.com
1,adm.com
2,progressive.com
3,oracle.com
4,telefonica.com/es


In [4]:
import swifter
import time

OUTPUT_CSV = "dns_pipeline_multi_ns.csv"

ERROR_CSV = "dns_pipeline_errors.csv"

INITIAL_INDEX = 0 # Número de domínios já processados anteriormente no caso de interrupção do script ou reinício do processamento.

# Configurações
BATCH_SIZE = 100
SLEEP_BETWEEN_BATCHES = 0  # segundos, se quiser pausar entre lotes
SAVE_MODE = 'a'  # append no CSV

# Carregar dataset
total_domains = len(dns_pipeline_df)

# Cabeçalho para salvar apenas no primeiro write
write_header = True

if os.path.exists(OUTPUT_CSV):
    output_pd = pd.read_csv(OUTPUT_CSV)
else:
    output_pd = pd.DataFrame(columns=['domain'])

display(output_pd.tail())
# Processamento em lotes
for i in range(INITIAL_INDEX, total_domains, BATCH_SIZE):
    batch_df = dns_pipeline_df.iloc[i:i+BATCH_SIZE].copy()

    print(f"\nProcessando domínios {i+1} a {min(i+BATCH_SIZE, total_domains)} de {total_domains}...\n")

    batch_results = []
    batch_errors = []

    for idx, domain in tqdm(enumerate(batch_df['domain']), total=len(batch_df), desc=f"Lote {i//BATCH_SIZE + 1}"):
        if domain in output_pd['domain'].values:
            print(f"Domínio {domain} já processado. Pulando...")
            continue

        print(f"Processando domínio: {domain}")
        ns_info_list, failed_entry = get_ns_info(domain)

        # monta as linhas de resultado (uma por NS encontrado)
        for ns_info in ns_info_list:
            enriched_row = batch_df.iloc[idx].to_dict()
            enriched_row.update(ns_info)
            batch_results.append(enriched_row)

        # registra erro se houver (inclusive quando não encontrou NS)
        if failed_entry:
            batch_errors.append(failed_entry)

    # Converter para DataFrame
    batch_results_df = pd.DataFrame(batch_results)

    # Salvar erros, se houver
    if batch_errors:
        errors_df = pd.DataFrame(batch_errors)
        if os.path.exists(ERROR_CSV):
            errors_df.to_csv(ERROR_CSV, mode='a', index=False, header=False)
        else:
            errors_df.to_csv(ERROR_CSV, index=False)
        print(f"⚠️ {len(batch_errors)} erros registrados em {ERROR_CSV}.")

    # Salvar incrementalmente
    batch_results_df.to_csv(OUTPUT_CSV, mode=SAVE_MODE, index=False, header=write_header)

    # Desligar header após o primeiro write
    write_header = False
    print(f"✅ Lote {i//BATCH_SIZE + 1} salvo. Total acumulado: {i + BATCH_SIZE} domínios (aproximadamente).\n")

    # Pausa opcional
    time.sleep(SLEEP_BETWEEN_BATCHES)


,domain



Processando domínios 1 a 100 de 216...



Lote 1:   0%|          | 0/100 [00:00<?, ?it/s]

Processando domínio: pg.com
Processando domínio: adm.com


Lote 1:   2%|▏         | 2/100 [00:00<00:10,  9.28it/s]

Processando domínio: progressive.com


... (316 outputs left out)

Lote 3:  94%|█████████▍| 15/16 [00:06<00:00,  2.94it/s]

Processando domínio: ubarhotels.com


Lote 3: 100%|██████████| 16/16 [00:06<00:00,  2.52it/s]

⚠️ 16 erros registrados em dns_pipeline_errors.csv.
✅ Lote 3 salvo. Total acumulado: 300 domínios (aproximadamente).

